In [1]:
# ==========================================
# CELDA 1: Librerías
# ==========================================
import os
import re
import json
import pickle
import warnings
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.preprocessing.sequence import pad_sequences

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
tf.get_logger().setLevel("ERROR")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


**Explicación:** En esta celda se importan las librerías necesarias para la fase final de inferencia, incluyendo el procesamiento del texto, la carga de artefactos guardados y la preparación de secuencias para predicción. A diferencia del notebook de modelado, aquí el enfoque está orientado exclusivamente al uso del modelo final entrenado.

In [2]:
# ==========================================
# CELDA 2: Rutas de archivos
# ==========================================
BASE_DIR = os.getcwd()

CONFIG_PATH = os.path.join(BASE_DIR, "config_modelo.json")
MODEL_ML_PATH = os.path.join(BASE_DIR, "mejor_modelo_ml.pkl")
TFIDF_PATH = os.path.join(BASE_DIR, "mejor_tfidf_vectorizer.pkl")

MODEL_DL_PATH = os.path.join(BASE_DIR, "mejor_modelo_dl.keras")
TOKENIZER_PATH = os.path.join(BASE_DIR, "tokenizer_fake_news.pkl")
MAXLEN_PATH = os.path.join(BASE_DIR, "max_length.txt")
THRESHOLD_PATH = os.path.join(BASE_DIR, "threshold.txt")

print("Rutas definidas.")

Rutas definidas.


**Explicación:** En esta sección se definen las rutas de acceso a los archivos generados durante la fase de modelado, incluyendo configuración, modelo entrenado, tokenizer y parámetros auxiliares. Centralizar estas rutas facilita la carga ordenada de artefactos y mejora la mantenibilidad del proceso de predicción.

In [3]:
# ==========================================
# CELDA 3: Funciones auxiliares
# ==========================================
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def recortar_texto(texto, max_palabras=300):
    palabras = str(texto).split()
    return " ".join(palabras[:max_palabras])

def preparar_entrada_prediccion(titulo="", texto="", max_palabras=300):
    titulo_limpio = clean_text(titulo)
    texto_limpio = clean_text(texto)
    texto_recortado = recortar_texto(texto_limpio, max_palabras=max_palabras)
    combinado = f"{titulo_limpio} {texto_recortado}".strip()
    return combinado

print("Funciones auxiliares listas.")

Funciones auxiliares listas.


**Explicación:** En esta celda se definen funciones auxiliares para limpiar el texto, controlar su longitud y construir la entrada final de predicción. Mantener el mismo preprocesamiento utilizado durante el entrenamiento garantiza consistencia entre la fase de modelado y la inferencia sobre nuevas noticias.

In [4]:
# ==========================================
# CELDA 4: Cargar configuración
# ==========================================
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config_modelo = json.load(f)

print("Configuración cargada correctamente.")
print(json.dumps(config_modelo, indent=4, ensure_ascii=False))

Configuración cargada correctamente.
{
    "mejor_modelo_nombre": "BiLSTM",
    "mejor_modelo_tipo": "Deep Learning",
    "f1_score_test": 0.9988555078683834,
    "feature": "content_clean = title_clean + text_trimmed_300",
    "vocab_size": 50000,
    "max_length": 300,
    "threshold": 0.6000000000000003
}


**Explicación e interpretación:** En esta etapa se carga la configuración almacenada del modelo final, la cual confirma que la **BiLSTM** fue la arquitectura con mejor desempeño durante la comparación. Además, se recuperan parámetros clave como el *max_length* y el *threshold*, necesarios para reproducir exactamente el mismo proceso de predicción definido en la fase de modelado.

In [5]:
# ==========================================
# CELDA 5: Cargar el mejor modelo
# ==========================================
mejor_modelo_nombre = config_modelo["mejor_modelo_nombre"]
mejor_modelo_tipo = config_modelo["mejor_modelo_tipo"]

modelo = None
vectorizer = None
tokenizer = None
max_length = None
threshold = None

if mejor_modelo_tipo == "Machine Learning":
    with open(MODEL_ML_PATH, "rb") as f:
        modelo = pickle.load(f)

    with open(TFIDF_PATH, "rb") as f:
        vectorizer = pickle.load(f)

    print(f"Modelo ML cargado: {mejor_modelo_nombre}")

elif mejor_modelo_tipo == "Deep Learning":
    modelo = tf.keras.models.load_model(MODEL_DL_PATH, compile=False)

    with open(TOKENIZER_PATH, "rb") as f:
        tokenizer = pickle.load(f)

    with open(MAXLEN_PATH, "r", encoding="utf-8") as f:
        max_length = int(f.read().strip())

    with open(THRESHOLD_PATH, "r", encoding="utf-8") as f:
        threshold = float(f.read().strip())

    print(f"Modelo DL cargado: {mejor_modelo_nombre}")
    print("max_length:", max_length)
    print("threshold:", threshold)

else:
    raise ValueError("Tipo de modelo no reconocido en config_modelo.json")

Modelo DL cargado: BiLSTM
max_length: 300
threshold: 0.6000000000000003


**Explicación:** En esta celda se realiza la carga dinámica del modelo con mejor desempeño a partir de la configuración almacenada. Dado que el modelo seleccionado fue la **BiLSTM**, se recuperan automáticamente la arquitectura entrenada, el *tokenizer*, el *max_length* y el *threshold*, asegurando que la inferencia utilice exactamente los mismos recursos definidos en la fase de entrenamiento.

In [6]:
# ==========================================
# CELDA 6: Función general de predicción
# ==========================================
def predecir_noticia(texto, titulo=""):
    if not isinstance(texto, str) or texto.strip() == "":
        return {"error": "Debes ingresar un texto válido."}

    entrada_modelo = preparar_entrada_prediccion(
        titulo=titulo,
        texto=texto,
        max_palabras=300
    )

    if entrada_modelo.strip() == "":
        return {"error": "La entrada procesada quedó vacía."}

    if mejor_modelo_tipo == "Machine Learning":
        entrada_vec = vectorizer.transform([entrada_modelo])
        clase = int(modelo.predict(entrada_vec)[0])

        if hasattr(modelo, "predict_proba"):
            prob_fake = float(modelo.predict_proba(entrada_vec)[0][1])
        else:
            prob_fake = None

    elif mejor_modelo_tipo == "Deep Learning":
        secuencia = tokenizer.texts_to_sequences([entrada_modelo])
        texto_pad = pad_sequences(
            secuencia,
            maxlen=max_length,
            truncating="post",
            padding="post"
        )

        prob_fake = float(modelo.predict(texto_pad, verbose=0)[0][0])
        clase = 1 if prob_fake >= threshold else 0

    else:
        return {"error": "Tipo de modelo no soportado."}

    etiqueta = "Fake News" if clase == 1 else "Noticia Real"

    resultado = {
        "modelo": mejor_modelo_nombre,
        "tipo_modelo": mejor_modelo_tipo,
        "titulo": titulo,
        "texto": texto,
        "entrada_procesada": entrada_modelo,
        "prediccion": etiqueta,
        "clase": clase
    }

    if prob_fake is not None:
        resultado["probabilidad_fake"] = round(prob_fake, 6)
        resultado["probabilidad_real"] = round(1 - prob_fake, 6)

    if threshold is not None:
        resultado["threshold"] = round(threshold, 6)

    return resultado

print("Función general de predicción lista.")

Función general de predicción lista.


**Explicación:** En esta celda se define la función general de predicción, encargada de validar la entrada, aplicar el mismo preprocesamiento del entrenamiento y generar la clasificación final de la noticia. Además de la etiqueta predicha, la función devuelve información complementaria como probabilidades y umbral de decisión, lo que facilita la interpretación del resultado obtenido.

In [7]:
# ==========================================
# CELDA 7: Prueba manual simple
# ==========================================
titulo_prueba = "Indonesia to buy $1.14 billion worth of Russian jets"
texto_prueba = """
JAKARTA (Reuters) - Indonesia will buy 11 Sukhoi fighter jets worth $1.14 billion from Russia in exchange for cash and Indonesian commodities, two cabinet ministers said on Tuesday.

The Southeast Asian country has pledged to ship up to $570 million worth of commodities in addition to cash to pay for the Sukhoi SU-35 fighter jets, which are expected to be delivered in stages starting in two years.

Indonesia is also trying to modernize its ageing air force after a string of military aviation accidents.
"""

resultado_prueba = predecir_noticia(texto=texto_prueba, titulo=titulo_prueba)
print(resultado_prueba)

{'modelo': 'BiLSTM', 'tipo_modelo': 'Deep Learning', 'titulo': 'Indonesia to buy $1.14 billion worth of Russian jets', 'texto': '\nJAKARTA (Reuters) - Indonesia will buy 11 Sukhoi fighter jets worth $1.14 billion from Russia in exchange for cash and Indonesian commodities, two cabinet ministers said on Tuesday.\n\nThe Southeast Asian country has pledged to ship up to $570 million worth of commodities in addition to cash to pay for the Sukhoi SU-35 fighter jets, which are expected to be delivered in stages starting in two years.\n\nIndonesia is also trying to modernize its ageing air force after a string of military aviation accidents.\n', 'entrada_procesada': 'indonesia to buy billion worth of russian jets jakarta reuters indonesia will buy sukhoi fighter jets worth billion from russia in exchange for cash and indonesian commodities two cabinet ministers said on tuesday the southeast asian country has pledged to ship up to million worth of commodities in addition to cash to pay for the

In [8]:
# ==========================================
# CELDA 8: Dataset sintético de prueba
# ==========================================
true_news_dataset_style = [
    {
        "titulo": "Canada and Mexico sign new trade coordination agreement",
        "texto": """
OTTAWA (Reuters) - Canada and Mexico signed a new agreement on Wednesday to strengthen trade coordination and streamline customs procedures across key manufacturing sectors, officials said.

The agreement focuses on automotive supply chains, agricultural exports, and technology services.

Government representatives said the measure is expected to improve cross-border competitiveness and reduce logistical delays over the coming year.
"""
    },
    {
        "titulo": "Japan unveils investment package to strengthen semiconductor production",
        "texto": """
TOKYO (Reuters) - Japan announced a new public-private investment initiative on Thursday aimed at expanding domestic semiconductor production and reducing reliance on overseas suppliers.

Officials said the package includes research funding, infrastructure development and tax incentives for strategic manufacturers.

The first projects are expected to begin before the end of the fiscal year.
"""
    },
    {
        "titulo": "European Union leaders discuss joint energy security measures",
        "texto": """
BRUSSELS (Reuters) - European Union leaders met on Friday to discuss new joint measures to improve regional energy security ahead of the winter season.

The proposals include expanded gas storage targets, renewable energy investments and emergency supply coordination mechanisms.

Diplomats said a formal framework could be approved in the next summit session.
"""
    },
    {
        "titulo": "Brazil central bank maintains benchmark interest rate",
        "texto": """
BRASILIA (Reuters) - Brazil's central bank kept its benchmark interest rate unchanged on Tuesday, citing easing inflation pressures and stable economic indicators.

Policymakers said recent labor market data and moderated consumer prices support a cautious monetary stance.

Analysts said the decision was largely in line with market expectations.
"""
    },
    {
        "titulo": "South Korea expands electric vehicle battery research partnership",
        "texto": """
SEOUL (Reuters) - South Korea announced a new research partnership between universities and private manufacturers to accelerate electric vehicle battery innovation.

The initiative will focus on longer battery life, improved safety standards and faster charging technologies.

Officials said the program aims to strengthen the country's position in the global EV supply chain.
"""
    }
]

fake_news_dataset_style = [
    {
        "titulo": "BREAKING: Congress approves law allowing social media monitoring of all private messages",
        "texto": """
A bombshell report claims Congress has secretly approved a law that allows federal agencies to monitor all private social media messages without a warrant.

Anonymous insiders say the system will begin operating nationwide next week under a new national security protocol.

Critics argue the move represents the most extreme expansion of digital surveillance in modern history.
"""
    },
    {
        "titulo": "SHOCK REPORT: Government plans mandatory digital ID chips for all citizens by next month",
        "texto": """
A leaked document allegedly reveals that government officials are preparing to require mandatory digital identification chips for every citizen starting next month.

Sources claim the chips will be linked to financial activity, health records and online access permissions.

Opponents say the measure is part of a long-term plan for total population control.
"""
    },
    {
        "titulo": "WATCH: Secret military program can now shut down the internet in entire cities",
        "texto": """
A stunning new exposé claims a classified military cyber program can instantly disable internet access across major cities.

The report says the technology has already been tested in several undisclosed urban areas.

Officials have denied the allegations, but insiders insist the infrastructure has been operational for years.
"""
    },
    {
        "titulo": "EXPOSED: Politicians caught using taxpayer money to fund private luxury islands",
        "texto": """
A shocking investigation alleges that several senior politicians have been using taxpayer funds to secretly purchase and maintain private luxury islands.

Documents shared online claim millions of dollars were redirected through shell organizations over the past decade.

Public outrage has exploded on social media as demands for criminal charges continue to grow.
"""
    },
    {
        "titulo": "ALERT: New school curriculum will require students to submit daily political loyalty scores",
        "texto": """
A viral report claims that a new education policy will force students to submit daily political loyalty scores through a government learning platform.

Parents and teachers say the program is designed to track ideological compliance from an early age.

Officials have not commented on the leaked screenshots circulating online.
"""
    }
]

print("Dataset sintético cargado.")

Dataset sintético cargado.


In [9]:
# ==========================================
# CELDA 9: Predicciones sobre dataset sintético
# ==========================================
resultados_sinteticos = []

# TRUE
for noticia in true_news_dataset_style:
    r = predecir_noticia(
        texto=noticia["texto"],
        titulo=noticia["titulo"]
    )

    resultados_sinteticos.append({
        "Clase esperada": "True",
        "Título": noticia["titulo"],
        "Modelo usado": r.get("modelo"),
        "Predicción": r.get("prediccion"),
        "Clase predicha": r.get("clase"),
        "Prob fake": r.get("probabilidad_fake", None),
        "Prob real": r.get("probabilidad_real", None)
    })

# FAKE
for noticia in fake_news_dataset_style:
    r = predecir_noticia(
        texto=noticia["texto"],
        titulo=noticia["titulo"]
    )

    resultados_sinteticos.append({
        "Clase esperada": "Fake",
        "Título": noticia["titulo"],
        "Modelo usado": r.get("modelo"),
        "Predicción": r.get("prediccion"),
        "Clase predicha": r.get("clase"),
        "Prob fake": r.get("probabilidad_fake", None),
        "Prob real": r.get("probabilidad_real", None)
    })

df_resultados_sinteticos = pd.DataFrame(resultados_sinteticos)
display(df_resultados_sinteticos)

,Clase esperada,Título,Modelo usado,Predicción,Clase predicha,Prob fake,Prob real
0,True,Canada and Mexico sign new trade coordination agreement,BiLSTM,Noticia Real,0,0.000202,0.999798
1,True,Japan unveils investment package to strengthen semiconductor production,BiLSTM,Noticia Real,0,0.000181,0.999819
2,True,European Union leaders discuss joint energy security measures,BiLSTM,Noticia Real,0,0.000123,0.999877
3,True,Brazil central bank maintains benchmark interest rate,BiLSTM,Noticia Real,0,0.000099,0.999901
4,True,South Korea expands electric vehicle battery research partnership,BiLSTM,Noticia Real,0,0.000384,0.999616
5,Fake,BREAKING: Congress approves law allowing social media monitoring of all private messages,BiLSTM,Fake News,1,0.999546,0.000454
6,Fake,SHOCK REPORT: Government plans mandatory digital ID chips for all citizens by next month,BiLSTM,Fake News,1,0.997262,0.002738
7,Fake,WATCH: Secret military program can now shut down the internet in entire cities,BiLSTM,Fake News,1,0.999957,0.000043
8,Fake,EXPOSED: Politicians caught using taxpayer money to fund private luxury islands,BiLSTM,Fake News,1,0.999911,0.000089
9,Fake,ALERT: New school curriculum will require students to submit daily political loyalty scores,BiLSTM,Fake News,1,0.999401,0.000599


**Explicación e interpretación:** En esta etapa se prueba la **BiLSTM** sobre un conjunto sintético de noticias verdaderas y falsas, con el fin de validar su comportamiento en escenarios controlados fuera del conjunto de entrenamiento. Los resultados muestran que el modelo clasificó correctamente los diez ejemplos, asignando probabilidades muy bajas a las noticias reales y muy altas a las noticias falsas, lo que refleja una alta consistencia en la fase de inferencia.

In [10]:
# ==========================================
# CELDA 10: Resumen de aciertos
# ==========================================
def normalizar_esperada(x):
    return 1 if x == "Fake" else 0

df_resultados_sinteticos["Clase esperada binaria"] = df_resultados_sinteticos["Clase esperada"].apply(normalizar_esperada)
df_resultados_sinteticos["Acierto"] = (
    df_resultados_sinteticos["Clase esperada binaria"] == df_resultados_sinteticos["Clase predicha"]
)

print("Cantidad de aciertos:", df_resultados_sinteticos["Acierto"].sum())
print("Cantidad de errores :", (~df_resultados_sinteticos["Acierto"]).sum())
print("Accuracy en pruebas sintéticas:", round(df_resultados_sinteticos["Acierto"].mean(), 4))

display(df_resultados_sinteticos)

Cantidad de aciertos: 10
Cantidad de errores : 0
Accuracy en pruebas sintéticas: 1.0


,Clase esperada,Título,Modelo usado,Predicción,Clase predicha,Prob fake,Prob real,Clase esperada binaria,Acierto
0,True,Canada and Mexico sign new trade coordination agreement,BiLSTM,Noticia Real,0,0.000202,0.999798,0,True
1,True,Japan unveils investment package to strengthen semiconductor production,BiLSTM,Noticia Real,0,0.000181,0.999819,0,True
2,True,European Union leaders discuss joint energy security measures,BiLSTM,Noticia Real,0,0.000123,0.999877,0,True
3,True,Brazil central bank maintains benchmark interest rate,BiLSTM,Noticia Real,0,0.000099,0.999901,0,True
4,True,South Korea expands electric vehicle battery research partnership,BiLSTM,Noticia Real,0,0.000384,0.999616,0,True
5,Fake,BREAKING: Congress approves law allowing social media monitoring of all private messages,BiLSTM,Fake News,1,0.999546,0.000454,1,True
6,Fake,SHOCK REPORT: Government plans mandatory digital ID chips for all citizens by next month,BiLSTM,Fake News,1,0.997262,0.002738,1,True
7,Fake,WATCH: Secret military program can now shut down the internet in entire cities,BiLSTM,Fake News,1,0.999957,0.000043,1,True
8,Fake,EXPOSED: Politicians caught using taxpayer money to fund private luxury islands,BiLSTM,Fake News,1,0.999911,0.000089,1,True
9,Fake,ALERT: New school curriculum will require students to submit daily political loyalty scores,BiLSTM,Fake News,1,0.999401,0.000599,1,True


**Interpretación:** El modelo obtuvo 10 aciertos de 10 casos evaluados en el conjunto sintético, alcanzando una exactitud de 1.0 en esta prueba controlada. Aunque estos resultados no sustituyen la evaluación formal sobre el conjunto de prueba, sí evidencian que la **BiLSTM** mantiene un comportamiento consistente al clasificar ejemplos nuevos construidos manualmente.

In [ ]:
# ==========================================
# CELDA 11: Uso manual - True inventada
# ==========================================
titulo_usuario = "Government announces new international trade agreement"
texto_usuario = """
Officials announced a new bilateral trade framework focused on technology exports,
regional investment and customs simplification. Analysts said the proposal could
strengthen economic cooperation over the next year.
"""

resultado_usuario = predecir_noticia(texto=texto_usuario, titulo=titulo_usuario)
print(resultado_usuario)

{'modelo': 'BiLSTM', 'tipo_modelo': 'Deep Learning', 'titulo': 'Government announces new international trade agreement', 'texto': '\nOfficials announced a new bilateral trade framework focused on technology exports,\nregional investment and customs simplification. Analysts said the proposal could\nstrengthen economic cooperation over the next year.\n', 'entrada_procesada': 'government announces new international trade agreement officials announced a new bilateral trade framework focused on technology exports regional investment and customs simplification analysts said the proposal could strengthen economic cooperation over the next year', 'prediccion': 'Noticia Real', 'clase': 0, 'probabilidad_fake': 0.024308, 'probabilidad_real': 0.975692, 'threshold': 0.6}


In [12]:
# ==========================================
# CELDA 12: Uso manual - True inventada
# ==========================================
titulo_usuario = "Germany launches new renewable energy investment program"

texto_usuario = """
BERLIN (Reuters) - Germany announced on Tuesday a new public-private investment
program aimed at expanding renewable energy infrastructure and modernizing
electricity distribution networks.

Officials said the initiative will focus on wind farms, solar energy storage
systems and regional grid resilience projects.

The first phase of funding is expected to begin later this year, with analysts
saying the measure could strengthen long-term energy security and reduce
industrial costs.
"""

resultado_usuario = predecir_noticia(
    texto=texto_usuario,
    titulo=titulo_usuario
)

print(resultado_usuario)

{'modelo': 'BiLSTM', 'tipo_modelo': 'Deep Learning', 'titulo': 'Germany launches new renewable energy investment program', 'texto': '\nBERLIN (Reuters) - Germany announced on Tuesday a new public-private investment\nprogram aimed at expanding renewable energy infrastructure and modernizing\nelectricity distribution networks.\n\nOfficials said the initiative will focus on wind farms, solar energy storage\nsystems and regional grid resilience projects.\n\nThe first phase of funding is expected to begin later this year, with analysts\nsaying the measure could strengthen long-term energy security and reduce\nindustrial costs.\n', 'entrada_procesada': 'germany launches new renewable energy investment program berlin reuters germany announced on tuesday a new public private investment program aimed at expanding renewable energy infrastructure and modernizing electricity distribution networks officials said the initiative will focus on wind farms solar energy storage systems and regional grid 

In [13]:
# ==========================================
# CELDA 13: Uso manual - Fake inventada
# ==========================================
titulo_usuario = "BREAKING: Government to require mandatory digital wallets for all citizens next month"

texto_usuario = """
A leaked internal report claims the government is preparing to require all citizens
to adopt mandatory digital wallets beginning next month.

According to anonymous officials, the wallets will be linked to tax payments,
health records, transportation access and online identity verification.

Critics say the initiative is part of a broader surveillance plan designed to
monitor personal spending habits and restrict access to financial services
for individuals who do not comply with new national standards.
"""

resultado_usuario = predecir_noticia(
    texto=texto_usuario,
    titulo=titulo_usuario
)

print(resultado_usuario)

{'modelo': 'BiLSTM', 'tipo_modelo': 'Deep Learning', 'titulo': 'BREAKING: Government to require mandatory digital wallets for all citizens next month', 'texto': '\nA leaked internal report claims the government is preparing to require all citizens\nto adopt mandatory digital wallets beginning next month.\n\nAccording to anonymous officials, the wallets will be linked to tax payments,\nhealth records, transportation access and online identity verification.\n\nCritics say the initiative is part of a broader surveillance plan designed to\nmonitor personal spending habits and restrict access to financial services\nfor individuals who do not comply with new national standards.\n', 'entrada_procesada': 'breaking government to require mandatory digital wallets for all citizens next month a leaked internal report claims the government is preparing to require all citizens to adopt mandatory digital wallets beginning next month according to anonymous officials the wallets will be linked to tax p